In [1]:
import os
import pickle
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# Charger les données

data_path = "../data/processed"
file_name = 'dataset_final.pkl'
with open(os.path.join(data_path, file_name), 'rb') as f:
    df = pickle.load(f)

In [4]:
# Charger un seul embedding

embeddings_path = "../data/embeddings"
# file_name = 'emb_sbert_multi.pkl'

# with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#     embedding = pickle.load(f)

In [5]:
# Charger tous les embeddings

# embeddings_path = "../data/embeddings"
# embeddings = {}

# for file_name in os.listdir(embeddings_path):
#     with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#         emb_name = file_name.split('.')[0]
#         embeddings[emb_name] = pickle.load(f)

### BERTopic

In [167]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
from hdbscan import HDBSCAN
import umap
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize

In [7]:
sentences = df["clean_comment"].tolist()

file_name_mutli = 'emb_sbert_multi.pkl'
with open(os.path.join(embeddings_path, file_name_mutli), 'rb') as f:
    embedding_multi = pickle.load(f)

file_name_fr = 'emb_sbert_fr.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_fr = pickle.load(f)

file_name_perf = 'emb_sbert_multi2.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_multi2 = pickle.load(f)

In [8]:
print(len(sentences))
print(embedding_multi.shape)
print(embedding_fr.shape)
print(embedding_multi2.shape)

15077
(15077, 384)
(15077, 768)
(15077, 768)


In [191]:
product_words = ['montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'reconditionné']

models = {}

for embedding, name in zip([embedding_fr, embedding_multi, embedding_multi2], ["français", "multilingue", "multilingue_2"]):
    print(name)

    vectorizer_model = CountVectorizer(
        stop_words=stopwords.words("french") + product_words,
        ngram_range=(1, 3),
        #min_df=2,
        max_df=0.95
    )
    # # ctfidf_model = ClassTfidfTransformer()

    hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=3
    )
    
    topic_model = BERTopic(
        language="french",
        vectorizer_model=vectorizer_model,
        hdbscan_model=hdbscan_model,
        verbose=False,
        nr_topics="auto"
        # ctfidf_model=ctfidf_model
    )
    
    topics, probs = topic_model.fit_transform(sentences, embedding)

    # topic_model.reduce_topics(sentences, nr_topics=30)
    
    topics = np.array(topics)
    
    # Identifier les gros clusters
    topic_sizes = pd.Series(topics).value_counts()
    large_topics = topic_sizes[topic_sizes > 2000].index  # seuil à ajuster selon dataset
    
    for t in large_topics:
        print("Topic :", t)
        # Récupérer les indices des documents dans le gros cluster
        idx = np.where(topics == t)[0]
        embeddings_subset = embedding[idx]
        
        # Diviser le gros cluster avec KMeans
        n_subclusters = int(len(idx) / 300)  # ~300 docs par sous-cluster
        print("Nombre de clusters créés par k-means :", n_subclusters)
        
        emb_norm = normalize(embeddings_subset)
        kmeans = MiniBatchKMeans(
            n_clusters=n_subclusters,
            batch_size=512,
            max_iter=200,
            n_init="auto"
        )
        sub_labels = kmeans.fit_predict(emb_norm)
        
        # Réassigner les labels dans `topics`
        max_topic_id = topics.max() + 1
        for i, doc_idx in enumerate(idx):
            topics[doc_idx] = max_topic_id + sub_labels[i]

        topic_model.update_topics(
            docs=sentences,
            topics=topics,
            vectorizer_model=topic_model.vectorizer_model,
            top_n_words=15
        )

    models[name] = topic_model

français


2025-11-19 16:26:29,749 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 26


2025-11-19 16:26:34,059 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : -1
Nombre de clusters créés par k-means : 15
multilingue


2025-11-19 16:26:51,097 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : -1
Nombre de clusters créés par k-means : 20
multilingue_2
Topic : 0
Nombre de clusters créés par k-means : 28


2025-11-19 16:27:04,192 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2025-11-19 16:27:08,598 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : -1
Nombre de clusters créés par k-means : 15


### Evaluation

#### Diversité

In [192]:
from itertools import chain

def topic_diversity(topic_model, top_n=10):
    """
    Calcule la diversité des topics d'un modèle BERTopic.
    """
    topics = topic_model.get_topics()

    # On extrait les mots uniquement (sans les scores)
    topic_words = []
    for topic_id, word_scores in topics.items():
        # BERTopic place les topics -1 et autres meta-topics, donc on ignore topic -1
        if topic_id == -1:
            continue
        top_words = [w for (w, score) in word_scores[:top_n]]
        topic_words.append(top_words)

    # Liste aplatie
    all_words = list(chain.from_iterable(topic_words))
    unique_words = set(all_words)

    return len(unique_words) / len(all_words)


# Calcul du score pour chaque modèle
diversity_scores = {}

for name, model in models.items():
    score = topic_diversity(model, top_n=10)
    diversity_scores[name] = score

diversity_scores

{'français': 0.5386861313868613,
 'multilingue': 0.6088397790055249,
 'multilingue_2': 0.5221153846153846}

#### Score de cohérence

In [193]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from collections import defaultdict
import numpy as np

for name, model in models.items():
    topics = model.topics_
    # topics = array final après KMeans
    docs_by_topic = defaultdict(list)
    for doc, t in zip(sentences, topics):
        if t != -1:
            docs_by_topic[t].append(doc)
    
    # Générer top words manuellement
    topic_words = []
    for t, docs in docs_by_topic.items():
        vec = TfidfVectorizer(stop_words=stopwords.words("french") + product_words, ngram_range=(1,3))
        X = vec.fit_transform(docs)
        feature_names = np.array(vec.get_feature_names_out())
        tfidf_sum = X.toarray().sum(axis=0)
        top_words = feature_names[np.argsort(tfidf_sum)[::-1]][:10].tolist()
        topic_words.append(top_words)
    
    # Tokenisation des documents
    tokenized_docs = [doc.lower().split() for doc in sentences]
    dictionary = Dictionary(tokenized_docs)
    
    # Calcul du score de cohérence
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    print(name, score)

français 0.44665827710076944
multilingue 0.4327499487686278
multilingue_2 0.44593031794908106


#### Embedding-based coherence score

In [194]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embedding_coherence(topic_model, embedder, top_n=10):
    """
    Calcule la cohérence basée sur les embeddings pour un modèle BERTopic.
    embedder : modèle sentence-transformers pour transformer les mots en vecteurs.
    """
    topics = topic_model.get_topics()
    scores = []

    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        top_words = [w for w, _ in word_scores[:top_n]]
        word_embeddings = embedder.encode(top_words)
        sim_matrix = cosine_similarity(word_embeddings)
        
        # On enlève la diagonale (sim = 1)
        n = len(top_words)
        if n > 1:
            sims = (sim_matrix.sum() - n) / (n*(n-1))  # moyenne des cosinus
            scores.append(sims)

    return np.mean(scores)

# Exemple avec un modèle SentenceTransformer
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for name, model in models.items():
    score = embedding_coherence(model, embedder, top_n=10)
    print(f"{name} - Embedding-based coherence: {score:.4f}")

français - Embedding-based coherence: 0.4092
multilingue - Embedding-based coherence: 0.3657
multilingue_2 - Embedding-based coherence: 0.4021


In [198]:
# for name, model in models.items():
#     with open(f"../models/topic_modeling/bertopic_{name}.pkl", "wb") as f:
#         pickle.dump(model, f)

In [204]:
# df["topics"] = models["multilingue_2"].topics_
# df.to_csv("./artifacts/bertopic/reviews_with_topics.csv", index=False, encoding="utf8")

### Visualisation

In [10]:
# CHARGER LES MODELES ENREGISTRES
# models = {}
# for name in ["français", "multilingue", "multilingue_2"]:
#     with open(os.path.join("../models/topic_modeling/", f"bertopic_{name}.pkl"), 'rb') as f:
#         models[name] = pickle.load(f)

In [11]:
all_topics_words = {}
topic_model = models["multilingue_2"]

for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue  # ignorer les outliers
    all_topics_words[topic_id] = [word for word, _ in topic_model.get_topic(topic_id)]

all_topics_words

{0: ['commande',
  'plus',
  'livraison',
  'colis',
  'site',
  'service',
  'vente',
  'client',
  'bien',
  'remboursement'],
 1: ['conforme',
  'livraison',
  'description',
  'rapide',
  'attentes',
  'conforme description',
  'produit conforme',
  'conformes',
  'produit',
  'prévue'],
 2: ['conforme',
  'attentes',
  'conforme attentes',
  'description',
  'produit conforme',
  'correspond',
  'attentes produit',
  'produit',
  'conforme description',
  'conforme commande'],
 3: ['rien dire',
  'rien',
  'dire',
  'rien redire',
  'redire',
  'parfait',
  'parfait rien',
  'rien dire rien',
  'dire rien',
  'parfait rien dire'],
 4: ['satisfaite',
  'très satisfaite',
  'contente',
  'achat très',
  'achat',
  'satisfaite achat',
  'achat très satisfaite',
  'très contente',
  'satisfaite achat très',
  'très satisfaite achat'],
 5: ['qualité',
  'déçue',
  'déçue qualité',
  'déçu',
  'photo',
  'déçu qualité',
  'correspondent',
  'articles',
  'photos',
  'correspond'],
 6: [

In [195]:
# topics trouvés
models["français"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,189,1_conforme_description_conformes_rapide,"[conforme, description, conformes, rapide, liv...",[livraison rapide . produit conforme à la desc...
1,2,149,2_rapide_livraison rapide_livraison_top,"[rapide, livraison rapide, livraison, top, par...","[très bon produit , livraison rapide, bon prod..."
2,3,140,3_livraison peu_peu_peu long_long,"[livraison peu, peu, peu long, long, longue, l...","[le délai de livraison est un peu long, délai ..."
3,4,69,4_merci_tout parfait_tout_merci tout,"[merci, tout parfait, tout, merci tout, bien m...","[tout est parfait , merci ., tout a été parfai..."
4,5,60,5_rapide_livraison rapide_conforme_très bien r...,"[rapide, livraison rapide, conforme, très bien...","[livraison rapide produit conforme, livraison ..."
...,...,...,...,...,...
132,133,219,133_livraison_arrivée_date_commande,"[livraison, arrivée, date, commande, temps, co...",NaN
133,134,79,134_merci_satisfaite_showroom_très,"[merci, satisfaite, showroom, très, satisfaite...",NaN
134,135,138,135_belles_jolies_très_confortables,"[belles, jolies, très, confortables, très bell...",NaN
135,136,189,136_satisfaite_contente_très_très satisfaite,"[satisfaite, contente, très, très satisfaite, ...",NaN


In [196]:
# topics trouvés
models["multilingue"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,1188,0_commande_livraison_long_service,"[commande, livraison, long, service, rembourse...","[délais de livraison trop long, en deux mois ,..."
1,1,484,1_rapide_livraison rapide_produit_livraison,"[rapide, livraison rapide, produit, livraison,...","[livraison rapide et produit conforme ., livra..."
2,2,300,2_site_plus_depuis_service,"[site, plus, depuis, service, client, jamais, ...","[client régulier , je constate de fortes dégra..."
3,3,272,3_article_articles_manque_manque article,"[article, articles, manque, manque article, re...","[il manque un article, il manque un article ....."
4,4,217,4_articles_article_articles conformes_satisfai...,"[articles, article, articles conformes, satisf...","[suis satisfaite des articles, très bel articl..."
...,...,...,...,...,...
176,176,344,176_service_client_service client_depuis,"[service, client, service client, depuis, plus...",NaN
177,177,388,177_colis_reçu_commandé_commande,"[colis, reçu, commandé, commande, fois, déçu, ...",NaN
178,178,179,178_très_contente_satisfaite_achat,"[très, contente, satisfaite, achat, très conte...",NaN
179,179,534,179_mail_service_commande_client,"[mail, service, commande, client, service clie...",NaN


In [205]:
# topics trouvés
# models["multilingue_2"].get_topic_info().to_csv("./artifacts/bertopic/topics_top_words.csv", index=False, encoding="utf8")
models["multilingue_2"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,148,1_livraison peu_peu_peu long_long,"[livraison peu, peu, peu long, long, longue, l...","[le délai de livraison est un peu long, délai ..."
1,2,103,2_rien dire_rien_dire_rien redire,"[rien dire, rien, dire, rien redire, redire, p...","[parfait rien a dire ..., parfait , rien a dir..."
2,3,84,3_rapide_livraison rapide_conforme_livraison,"[rapide, livraison rapide, conforme, livraison...","[livraison rapide et produit conforme, livrais..."
3,4,64,4_attentes_conforme attentes_conforme_attentes...,"[attentes, conforme attentes, conforme, attent...","[conforme a mes attentes, conforme à mes atten..."
4,5,53,5_rien_rien dire_dire_rien redire,"[rien, rien dire, dire, rien redire, redire, r...","[très bien rien à dire, rien n ' a dire, je ri..."
...,...,...,...,...,...
99,100,219,100_rapide_conforme_livraison rapide_conformes,"[rapide, conforme, livraison rapide, conformes...",NaN
100,101,235,101_livraison_long_trop_rapide,"[livraison, long, trop, rapide, produit, livra...",NaN
101,102,587,102_site_plus_commande_colis,"[site, plus, commande, colis, service, vente, ...",NaN
102,103,219,103_satisfaite_commande_contente_très,"[satisfaite, commande, contente, très, très sa...",NaN


In [251]:
# Mots-clés associés à un topic
models["français"].get_topic(0)

[('commande', np.float64(0.010340701984429202)),
 ('plus', np.float64(0.00986959838147149)),
 ('colis', np.float64(0.007717754654670451)),
 ('site', np.float64(0.007657288166145758)),
 ('service', np.float64(0.007449170381609198)),
 ('vente', np.float64(0.0067841269904930415)),
 ('livraison', np.float64(0.00670084095999104)),
 ('client', np.float64(0.00665961608731454)),
 ('remboursement', np.float64(0.0059745462739572855)),
 ('après', np.float64(0.005613513495396216))]

In [252]:
# Mots-clés associés à un topic
models["multilingue"].get_topic(0)

[('livraison', np.float64(0.01209649178161709)),
 ('commande', np.float64(0.01157781764070824)),
 ('très', np.float64(0.011430530478093958)),
 ('plus', np.float64(0.00938488756488948)),
 ('site', np.float64(0.007777954416698765)),
 ('colis', np.float64(0.007690737782841499)),
 ('bien', np.float64(0.007397506768599091)),
 ('service', np.float64(0.006689339871293234)),
 ('tout', np.float64(0.006139802726551881)),
 ('qualité', np.float64(0.006094333420782947))]

In [253]:
fig1 = models["français"].visualize_topics(top_n_topics=10)
fig2 = models["multilingue"].visualize_topics(top_n_topics=10)

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Modèle 1 : Français", "Modèle 2 : Multilingue")
)

for trace in fig1['data']:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig2['data']:
    combined_fig.add_trace(trace, row=1, col=2)

combined_fig.update_layout(
    title_text="Répartition des topics",
    showlegend=False,
    height=600,
    width=1000
)

combined_fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
models["français"].visualize_barchart(top_n_topics=10)

In [ ]:
models["multilingue"].visualize_barchart(top_n_topics=10)

In [ ]:
models["multilingue"].visualize_hierarchy()

In [ ]:
models["multilingue"].visualize_hierarchy()